# 🎨 [제01화] 새벽 대청 — Colab SDXL 원클릭 서정 삽화 생성기

- **작품집/프로젝트**: 『시화집』
- **작품 정보**: [제01화] 「새벽 대청」
- **생성 규격**: 출판용 고해상도 (768x1024, 300 DPI 대응) 2종 (시안 A, 시안 B)
- **환경**: Google Colab 무료 T4 GPU (15GB VRAM)

> 💡 **[초간단 원클릭 실행 안내]**
> 상단 메뉴에서 **[런타임] → [모두 실행]**을 누르면 자동으로 패키지 설치부터 초고속 AI 렌더링, 그리고 브라우저로 **`01_새벽_대청_시안A.jpg`, `01_새벽_대청_시안B.jpg` 2가지 시안 자동 다운로드**까지 한 번에 완료됩니다.
> 다운로드된 그림 중 마음에 드는 시안을 스토리북 스튜디오나 뷰어에 등록하시면 됩니다.

In [ ]:
# [1단계] GPU 하드웨어 가속기 점검 (Google Colab T4 16GB VRAM)
!nvidia-smi

import torch
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"연결된 GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM 총량: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    raise SystemError("GPU가 감지되지 않았습니다. [런타임] -> [런타임 유형 변경]에서 T4 GPU를 선택해 주세요!")

In [ ]:
# [2단계] 초고속 렌더링 라이브러리 설치 (diffusers, transformers, accelerate, torchao)
!pip install -q --upgrade diffusers transformers accelerate safetensors torchao
print("✅ 필수 패키지 설치 완료!")

In [ ]:
# @title [3단계] 삽화 메타데이터 및 프롬프트 설정 (동적 조립 모듈)
# @markdown 스토리북 스튜디오에서 생성된 작품 정보가 사전에 설정되어 있습니다.
# @markdown 필요시 다른 프로젝트에 맞게 값을 변경하고 3~5단계를 다시 실행할 수 있습니다.
import os

EPISODE_NO = "제01화"  # @param {type:"string"}
WORK_TITLE = "새벽 대청"  # @param {type:"string"}
FILE_A = "01_새벽_대청_시안A.jpg"  # @param {type:"string"}
FILE_B = "01_새벽_대청_시안B.jpg"  # @param {type:"string"}
IMAGE_WIDTH = 768  # @param {type:"integer"}
IMAGE_HEIGHT = 1024  # @param {type:"integer"}
STEPS = 4  # @param {type:"integer"}

# 1안 (서정적 여백 중심)
PROMPT_A = """새벽 토방마루와 마당의 싸리비 흔적, tranquil poetic negative space, minimalist composition, korean lyric watercolor painting, delicate brush strokes, soft watercolor wash, poetic negative space, gentle morning light, nostalgic mood, masterpiece, highly detailed, 300 dpi illustration"""

# 2안 (따스한 정경 디테일 중심)
PROMPT_B = """새벽 토방마루와 마당의 싸리비 흔적, warm nostalgic ambiance, intimate poetic harmony, rich atmospheric details, korean lyric watercolor painting, delicate brush strokes, soft watercolor wash, poetic negative space, gentle morning light, nostalgic mood, masterpiece, highly detailed, 300 dpi illustration"""

NEGATIVE_PROMPT = """ugly, deformed, disfigured, poor anatomy, bad hands, blurry, low quality, oversaturated, text, watermark, signature, modern electronics, photo frame"""

print(f"✅ 3단계: [{EPISODE_NO}] {WORK_TITLE} 삽화 프롬프트 준비 완료!")
print(f" - 시안 A: {FILE_A}")
print(f" - 시안 B: {FILE_B}")

In [ ]:
# [4단계] SDXL-Lightning 파이프라인 로드 및 2가지 시안 렌더링 (단 5~10초 소요)
import torch
from diffusers import StableDiffusionXLPipeline, UNet2DConditionModel, EulerDiscreteScheduler
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

base_model = "stabilityai/stable-diffusion-xl-base-1.0"
repo_lightning = "ByteDance/SDXL-Lightning"
ckpt = f"sdxl_lightning_{STEPS}step_unet.safetensors"

print("🎨 초고속 SDXL Lightning 엔진을 로드합니다...")
# 4스텝 고속 UNet 로드
unet = UNet2DConditionModel.from_config(base_model, subfolder="unet").to("cuda", torch.float16)
unet.load_state_dict(load_file(hf_hub_download(repo_lightning, ckpt), device="cuda"))

# 파이프라인 조립
pipe = StableDiffusionXLPipeline.from_pretrained(
    base_model,
    unet=unet,
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

# Lightning 권장 스케줄러 설정
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing="trailing")

os.makedirs("/content/output", exist_ok=True)

print(f"\n🖼️ [시안 A: 서정적 여백 중심] 렌더링 중...")
gen_a = torch.Generator("cuda").manual_seed(42)
img_a = pipe(
    prompt=PROMPT_A,
    negative_prompt=NEGATIVE_PROMPT,
    num_inference_steps=STEPS,
    guidance_scale=0.0,
    width=IMAGE_WIDTH,
    height=IMAGE_HEIGHT,
    generator=gen_a
).images[0]
path_a = os.path.join("/content", FILE_A)
img_a.save(path_a, quality=95)
print(f"✓ 시안 A 저장 완료: {path_a}")

print(f"\n🖼️ [시안 B: 따스한 정경 디테일] 렌더링 중...")
gen_b = torch.Generator("cuda").manual_seed(1004)
img_b = pipe(
    prompt=PROMPT_B,
    negative_prompt=NEGATIVE_PROMPT,
    num_inference_steps=STEPS,
    guidance_scale=0.0,
    width=IMAGE_WIDTH,
    height=IMAGE_HEIGHT,
    generator=gen_b
).images[0]
path_b = os.path.join("/content", FILE_B)
img_b.save(path_b, quality=95)
print(f"✓ 시안 B 저장 완료: {path_b}")
print("\n🎉 2가지 삽화 시안 생성이 모두 성공적으로 완료되었습니다!")

In [ ]:
# [5단계] 2가지 시안 나란히 미리보기 및 컴퓨터로 자동 다운로드
import os
from IPython.display import Image, display
from google.colab import files

path_a = os.path.join("/content", FILE_A)
path_b = os.path.join("/content", FILE_B)

print("=" * 65)
print(f"🎨 [{EPISODE_NO}] {WORK_TITLE} — 2가지 삽화 시안 미리보기")
print("=" * 65)

if os.path.exists(path_a):
    print(f"\n[시안 A: 서정적 여백 중심] ({FILE_A})")
    display(Image(path_a, width=380))

if os.path.exists(path_b):
    print(f"\n[시안 B: 따스한 정경 디테일] ({FILE_B})")
    display(Image(path_b, width=380))

print("\n📥 대표님 PC의 [다운로드] 폴더로 2가지 삽화 시안 자동 다운로드를 시작합니다...")
if os.path.exists(path_a):
    files.download(path_a)
if os.path.exists(path_b):
    files.download(path_b)

print(f"💡 다운로드된 그림 중 마음에 드는 파일을 스토리북 스튜디오의 [그림 파일 교체] 또는 드롭존에 등록하세요!")